# Decision Analysis Teaching Notebook (Car Example + Reusable Template)

This notebook has two goals:
1. Teach the linear value model workflow using a car decision example.
2. Act as a template you can reuse for a different domain by editing only the input blocks.

The workflow is always: criteria -> alternatives/raw data -> value functions -> weights -> scoring/results.

## 0) Setup and imports

Run the next two cells first.

### Google Colab

Before sharing this notebook, replace `https://github.com/OWNER/REPOSITORY.git` in the first code cell with this project's GitHub clone URL. When students open the notebook in Colab, that cell clones the repository into `/content/decision-core` and installs the dependencies listed in `requirements.txt`.

For local use, start Jupyter from the repository root; the same cells use the local checkout without cloning.

Keep the import section unchanged unless your project layout differs.

In [ ]:
# Google Colab setup
# Replace this URL after publishing the repository on GitHub.
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/OWNER/REPOSITORY.git"
REPOSITORY_DIRECTORY = Path("/content/decision-core")

if "google.colab" in sys.modules:
    if "OWNER/REPOSITORY" in REPOSITORY_URL:
        raise RuntimeError(
            "Set REPOSITORY_URL to this project's GitHub clone URL, then rerun this cell."
        )

    if not REPOSITORY_DIRECTORY.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIRECTORY)],
            check=True,
        )

    PROJECT_ROOT = REPOSITORY_DIRECTORY
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"],
        check=True,
        cwd=PROJECT_ROOT,
    )
else:
    PROJECT_ROOT = Path.cwd().resolve()

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", Path.cwd().resolve())
DECISION_CORE_SRC = PROJECT_ROOT / "src" / "decision-core"
DECISION_PLOTS_SRC = PROJECT_ROOT / "src" / "decision-plots"
IMPORT_DATA_FRAMES_SRC = PROJECT_ROOT / "src" / "import-data-frames"

for source_path in (DECISION_CORE_SRC, DECISION_PLOTS_SRC, IMPORT_DATA_FRAMES_SRC):
    source_path_str = str(source_path)
    if source_path_str not in sys.path:
        sys.path.insert(0, source_path_str)

from decision_core import TotalValueModel
from import_data_frames import (
    build_alternatives_from_data_frame,
    build_criteria_from_data_frame,
    build_criterion_weights_from_data_frame,
    build_decision_problem,
    build_default_feasibility_predicate_registry,
    build_feasibility_rules_from_data_frame,
    build_value_functions_from_specs,
    validate_input_shapes,
)

try:
    from decision_plots import DecisionAnalysisStackedBarPlotter
    HAS_DECISION_PLOTS = True
except ImportError:
    HAS_DECISION_PLOTS = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print("Imports loaded. decision_plots available:", HAS_DECISION_PLOTS)

Imports loaded. decision_plots available: True


## 1) Editable input block: criteria

Students: edit this table for your own problem.

Required columns:
- `id`, `name`, `kind`, `min_value`, `max_value`, `allowed_values`
- `allowed_values` should be a Python list for `ordinal`/`categorical`, otherwise `None`.

In [2]:
criteria_df = pd.DataFrame([
    {"id": "stopping_distance", "name": "Stopping Distance", "kind": "continuous", "min_value": 0.0, "max_value": None, "allowed_values": None},
    {"id": "airbags", "name": "Airbags", "kind": "integer", "min_value": 0, "max_value": None, "allowed_values": None},
    {"id": "rollover_rating", "name": "Rollover Rating", "kind": "ordinal", "min_value": None, "max_value": None, "allowed_values": [1, 2, 3, 4, 5]},
    {"id": "apps", "name": "Apps", "kind": "categorical", "min_value": None, "max_value": None, "allowed_values": ["None", "Subscription", "Wired Car Play", "Wireless Car Play"]},
    {"id": "cargo_space", "name": "Cargo space", "kind": "continuous", "min_value": 0.0, "max_value": None, "allowed_values": None},
    {"id": "traction_control", "name": "Traction Control", "kind": "boolean", "min_value": None, "max_value": None, "allowed_values": None},
    {"id": "rear_legroom", "name": "Rear Legroom", "kind": "continuous", "min_value": 0.0, "max_value": None, "allowed_values": None},
    {"id": "crash_ratings", "name": "Crash Ratings", "kind": "ordinal", "min_value": None, "max_value": None, "allowed_values": [1, 2, 3, 4, 5]},
])

criteria_df

,id,name,kind,min_value,max_value,allowed_values
0,stopping_distance,Stopping Distance,continuous,0.0,None,None
1,airbags,Airbags,integer,0.0,None,None
2,rollover_rating,Rollover Rating,ordinal,NaN,None,"[1, 2, 3, 4, 5]"
3,apps,Apps,categorical,NaN,None,"[None, Subscription, Wired Car Play, Wireless ..."
4,cargo_space,Cargo space,continuous,0.0,None,None
5,traction_control,Traction Control,boolean,NaN,None,None
6,rear_legroom,Rear Legroom,continuous,0.0,None,None
7,crash_ratings,Crash Ratings,ordinal,NaN,None,"[1, 2, 3, 4, 5]"


## 2) Editable input block: alternatives + raw measurements

Students: edit rows and values. Keep `id`, `name`, and criterion columns aligned with `criteria_df['id']`.

In [3]:
raw_df = pd.DataFrame([
    {"id": "cadillac_srx", "name": "Cadillac SRX Luxury 4WD", "stopping_distance": 119.0, "airbags": 6, "rollover_rating": 4, "apps": "Subscription", "cargo_space": 29.8, "traction_control": True, "rear_legroom": 36.0, "crash_ratings": 5},
    {"id": "acura_rdx", "name": "Acura RDX SH 4WD", "stopping_distance": 125.0, "airbags": 6, "rollover_rating": 4, "apps": "Wireless Car Play", "cargo_space": 28.0, "traction_control": True, "rear_legroom": 37.6, "crash_ratings": 5},
    {"id": "honda_crv", "name": "Honda CR-V EX-L 4WD", "stopping_distance": 119.0, "airbags": 6, "rollover_rating": 4, "apps": "Wireless Car Play", "cargo_space": 35.7, "traction_control": True, "rear_legroom": 38.5, "crash_ratings": 5},
    {"id": "toyota_rav4", "name": "Toyota RAV4", "stopping_distance": 130.0, "airbags": 6, "rollover_rating": 4, "apps": "Subscription", "cargo_space": 36.4, "traction_control": True, "rear_legroom": 38.3, "crash_ratings": 5},
    {"id": "volvo_xc60", "name": "Volvo XC-60", "stopping_distance": 123.0, "airbags": 6, "rollover_rating": 4, "apps": "Wireless Car Play", "cargo_space": 30.8, "traction_control": True, "rear_legroom": 36.4, "crash_ratings": 5},
    {"id": "lexus_rx_350", "name": "Lexus RX 350 AWD", "stopping_distance": 130.0, "airbags": 10, "rollover_rating": 4, "apps": "Wireless Car Play", "cargo_space": 40.0, "traction_control": True, "rear_legroom": 36.8, "crash_ratings": 5},
    {"id": "mitsubishi_outlander", "name": "Mitsubishi Outlander", "stopping_distance": 141.0, "airbags": 8, "rollover_rating": 4, "apps": "Wireless Car Play", "cargo_space": 36.2, "traction_control": True, "rear_legroom": 39.6, "crash_ratings": 5},
    {"id": "lexus_rx_350_2009", "name": "Lexus RX 350 AWD (2009)", "stopping_distance": 130.0, "airbags": 10, "rollover_rating": 4, "apps": "Wireless Car Play", "cargo_space": 40.0, "traction_control": True, "rear_legroom": 36.8, "crash_ratings": 5},
    {"id": "honda_crv_2009", "name": "Honda CR-V EX-L 4WD (2009)", "stopping_distance": 119.0, "airbags": 6, "rollover_rating": 4, "apps": "Subscription", "cargo_space": 35.7, "traction_control": True, "rear_legroom": 38.5, "crash_ratings": 5},
    {"id": "honda_odyssey", "name": "Honda Odyssey EX-L", "stopping_distance": 147.0, "airbags": 8, "rollover_rating": 4, "apps": "None", "cargo_space": 38.4, "traction_control": True, "rear_legroom": 40.9, "crash_ratings": 5},
])

raw_df

,id,name,stopping_distance,airbags,rollover_rating,apps,cargo_space,traction_control,rear_legroom,crash_ratings
0,cadillac_srx,Cadillac SRX Luxury 4WD,119.0,6,4,Subscription,29.8,True,36.0,5
1,acura_rdx,Acura RDX SH 4WD,125.0,6,4,Wireless Car Play,28.0,True,37.6,5
2,honda_crv,Honda CR-V EX-L 4WD,119.0,6,4,Wireless Car Play,35.7,True,38.5,5
3,toyota_rav4,Toyota RAV4,130.0,6,4,Subscription,36.4,True,38.3,5
4,volvo_xc60,Volvo XC-60,123.0,6,4,Wireless Car Play,30.8,True,36.4,5
5,lexus_rx_350,Lexus RX 350 AWD,130.0,10,4,Wireless Car Play,40.0,True,36.8,5
6,mitsubishi_outlander,Mitsubishi Outlander,141.0,8,4,Wireless Car Play,36.2,True,39.6,5
7,lexus_rx_350_2009,Lexus RX 350 AWD (2009),130.0,10,4,Wireless Car Play,40.0,True,36.8,5
8,honda_crv_2009,Honda CR-V EX-L 4WD (2009),119.0,6,4,Subscription,35.7,True,38.5,5
9,honda_odyssey,Honda Odyssey EX-L,147.0,8,4,None,38.4,True,40.9,5


## 3) Editable input block: value-function specs

Students: edit only the values for your domain while keeping the overall structure the same.

In [4]:
value_function_specs = {
    "stopping_distance": {"type": "linear_decreasing", "min_raw_value": 100.0, "max_raw_value": 150.0},
    "airbags": {"type": "piecewise_linear", "points": [(2.0, 10.0), (4.0, 20.0), (6.0, 80.0), (8.0, 100.0), (10.0, 100.0)]},
    "rollover_rating": {"type": "mapping", "value_mapping": {1: 0.0, 2: 0.0, 3: 10.0, 4: 50.0, 5: 100.0}},
    "apps": {"type": "mapping", "value_mapping": {"None": 0.0, "Subscription": 30.0, "Wired Car Play": 60.0, "Wireless Car Play": 100.0}},
    "cargo_space": {"type": "linear_increasing", "min_raw_value": 25.0, "max_raw_value": 50.0},
    "traction_control": {"type": "mapping", "value_mapping": {False: 0.0, True: 100.0}},
    "rear_legroom": {"type": "linear_increasing", "min_raw_value": 30.0, "max_raw_value": 40.0},
    "crash_ratings": {"type": "mapping", "value_mapping": {1: 0.0, 2: 0.0, 3: 10.0, 4: 50.0, 5: 100.0}},
}

value_function_specs

{'stopping_distance': {'type': 'linear_decreasing',
  'min_raw_value': 100.0,
  'max_raw_value': 150.0},
 'airbags': {'type': 'piecewise_linear',
  'points': [(2.0, 10.0),
   (4.0, 20.0),
   (6.0, 80.0),
   (8.0, 100.0),
   (10.0, 100.0)]},
 'rollover_rating': {'type': 'mapping',
  'value_mapping': {1: 0.0, 2: 0.0, 3: 10.0, 4: 50.0, 5: 100.0}},
 'apps': {'type': 'mapping',
  'value_mapping': {'None': 0.0,
   'Subscription': 30.0,
   'Wired Car Play': 60.0,
   'Wireless Car Play': 100.0}},
 'cargo_space': {'type': 'linear_increasing',
  'min_raw_value': 25.0,
  'max_raw_value': 50.0},
 'traction_control': {'type': 'mapping',
  'value_mapping': {False: 0.0, True: 100.0}},
 'rear_legroom': {'type': 'linear_increasing',
  'min_raw_value': 30.0,
  'max_raw_value': 40.0},
 'crash_ratings': {'type': 'mapping',
  'value_mapping': {1: 0.0, 2: 0.0, 3: 10.0, 4: 50.0, 5: 100.0}}}

## 4) Editable input block: weights

Students: change the `weight` values to express your preferences.

In [5]:
weights_df = pd.DataFrame([
    {"criterion_id": "stopping_distance", "weight": 100.0},
    {"criterion_id": "airbags", "weight": 100.0},
    {"criterion_id": "rollover_rating", "weight": 75.0},
    {"criterion_id": "apps", "weight": 75.0},
    {"criterion_id": "cargo_space", "weight": 75.0},
    {"criterion_id": "traction_control", "weight": 0.0},
    {"criterion_id": "rear_legroom", "weight": 50.0},
    {"criterion_id": "crash_ratings", "weight": 50.0},
])

weights_df

,criterion_id,weight
0,stopping_distance,100.0
1,airbags,100.0
2,rollover_rating,75.0
3,apps,75.0
4,cargo_space,75.0
5,traction_control,0.0
6,rear_legroom,50.0
7,crash_ratings,50.0


## 5) Editable input block: feasibility rules

Define feasibility rules in data-frame form using the adapter predicate kinds.

Required columns:
- `id`, `name`, `criterion_id`, `predicate_kind`, `predicate_param`
- Built-in predicate kinds: `requires_true`, `not_equal`, `gte`.


In [6]:
feasibility_df = pd.DataFrame([
    {
        "id": "requires_traction_control",
        "name": "Requires traction control",
        "criterion_id": "traction_control",
        "predicate_kind": "requires_true",
        "predicate_param": None,
    },
    {
        "id": "apps_must_not_be_none",
        "name": "Apps must not be None",
        "criterion_id": "apps",
        "predicate_kind": "not_equal",
        "predicate_param": "None",
    },
    {
        "id": "rollover_rating_minimum",
        "name": "Rollover rating must be at least 4",
        "criterion_id": "rollover_rating",
        "predicate_kind": "gte",
        "predicate_param": 4.0,
    },
])

feasibility_df


,id,name,criterion_id,predicate_kind,predicate_param
0,requires_traction_control,Requires traction control,traction_control,requires_true,None
1,apps_must_not_be_none,Apps must not be None,apps,not_equal,None
2,rollover_rating_minimum,Rollover rating must be at least 4,rollover_rating,gte,4.0


## 5) Input-port validation checkpoint

Use the `import_data_frames` input-port helper to validate basic table shape/alignment before object construction.


In [7]:
shape_errors = validate_input_shapes(
    criteria_df,
    raw_df,
    weights_df,
    value_function_specs,
    feasibility_df,
    build_default_feasibility_predicate_registry(),
)

if not shape_errors:
    print("✅ Input-shape checkpoint passed.")
else:
    display(pd.DataFrame(shape_errors).sort_values(["stage", "row_or_key", "field"]).reset_index(drop=True))
    raise ValueError("Input-shape checkpoint failed. Review the error table above, update your input blocks, and rerun.")


✅ Input-shape checkpoint passed.


In [8]:
# Additional validation details are handled by the import_data_frames helpers.


## 6) Build core objects from data frames

These constructors come from the `import_data_frames` adapter and map the input data frames/specs to core model objects.


In [9]:
criteria = build_criteria_from_data_frame(criteria_df)
alternatives = build_alternatives_from_data_frame(criteria, raw_df)
decision_problem = build_decision_problem(
    criteria,
    alternatives,
    problem_id="car-decision-template",
    problem_name="Car Decision Template",
)

print(f"Built DecisionProblem with {len(criteria)} criteria and {len(alternatives)} alternatives.")


Built DecisionProblem with 8 criteria and 10 alternatives.


## 7) Build value functions and total-value model

Use adapter helpers for value functions and criterion weights, then construct the core total-value model.


In [10]:
value_functions = build_value_functions_from_specs(value_function_specs)
criterion_weights = build_criterion_weights_from_data_frame(weights_df)
feasibility_predicate_registry = build_default_feasibility_predicate_registry()
feasibility_rules = build_feasibility_rules_from_data_frame(feasibility_df, feasibility_predicate_registry)

total_value_model = TotalValueModel(
    id="template-total-value-model",
    name="Template Total Value Model",
    decision_problem=decision_problem,
    criterion_value_functions=value_functions,
    criterion_weights=criterion_weights,
    feasibility_rules=feasibility_rules,
)


## 8) Inspect model artifacts (optional)

Quick check of the built criterion weights before scoring.


In [11]:
display(
    pd.DataFrame(
        [{"criterion_id": criterion_id, "weight": weight} for criterion_id, weight in criterion_weights.items()]
    )
)


,criterion_id,weight
0,stopping_distance,100.0
1,airbags,100.0
2,rollover_rating,75.0
3,apps,75.0
4,cargo_space,75.0
5,traction_control,0.0
6,rear_legroom,50.0
7,crash_ratings,50.0


## 9) Score alternatives and build result tables

This produces per-criterion value scores, total scores, and ranking.


In [12]:
decision_result = total_value_model.score_decision_problem()

alternative_name_by_id = {alternative.id: alternative.name for alternative in decision_problem.alternatives}

rows: list[dict[str, object]] = []
for result_row in decision_result.rows:
    row = {
        "alternative_id": result_row.alternative_id,
        "alternative_name": alternative_name_by_id[result_row.alternative_id],
        **result_row.criterion_value_scores,
        "total_score": result_row.total_score,
    }
    rows.append(row)

scores_df = pd.DataFrame(rows).sort_values("total_score", ascending=False).reset_index(drop=True)
scores_df.insert(0, "rank", range(1, len(scores_df) + 1))

display(scores_df)


,rank,alternative_id,alternative_name,stopping_distance,airbags,rollover_rating,apps,cargo_space,traction_control,rear_legroom,crash_ratings,total_score
0,1,lexus_rx_350,Lexus RX 350 AWD,40.0,100.0,50.0,100.0,60.0,100.0,68.0,100.0,72.666667
1,2,lexus_rx_350_2009,Lexus RX 350 AWD (2009),40.0,100.0,50.0,100.0,60.0,100.0,68.0,100.0,72.666667
2,3,honda_crv,Honda CR-V EX-L 4WD,62.0,80.0,50.0,100.0,42.8,100.0,85.0,100.0,72.209524
3,4,mitsubishi_outlander,Mitsubishi Outlander,18.0,100.0,50.0,100.0,44.8,100.0,96.0,100.0,68.971429
4,5,volvo_xc60,Volvo XC-60,54.0,80.0,50.0,100.0,23.2,100.0,64.0,100.0,65.885714
5,6,acura_rdx,Acura RDX SH 4WD,50.0,80.0,50.0,100.0,12.0,100.0,76.0,100.0,64.666667
6,7,honda_crv_2009,Honda CR-V EX-L 4WD (2009),62.0,80.0,50.0,30.0,42.8,100.0,85.0,100.0,62.209524
7,8,toyota_rav4,Toyota RAV4,40.0,80.0,50.0,30.0,45.6,100.0,83.0,100.0,58.228571
8,9,cadillac_srx,Cadillac SRX Luxury 4WD,62.0,80.0,50.0,30.0,19.2,100.0,60.0,100.0,56.457143
9,10,honda_odyssey,Honda Odyssey EX-L,6.0,100.0,50.0,0.0,53.6,100.0,100.0,100.0,54.038095


## 10) Criterion-contribution view (weighted explanation)

This table explains each total score as the sum of weighted criterion contributions.


In [13]:
normalized_weights = total_value_model.normalized_weights()

contribution_rows: list[dict[str, object]] = []
for result_row in decision_result.rows:
    row = {
        "alternative_id": result_row.alternative_id,
        "alternative_name": alternative_name_by_id[result_row.alternative_id],
    }

    total = 0.0
    for criterion_id, value_score in result_row.criterion_value_scores.items():
        contribution = value_score * normalized_weights[criterion_id]
        row[criterion_id] = contribution
        total += contribution

    row["total_score_from_contributions"] = total
    contribution_rows.append(row)

contributions_df = pd.DataFrame(contribution_rows).sort_values("total_score_from_contributions", ascending=False).reset_index(drop=True)
display(contributions_df)

,alternative_id,alternative_name,stopping_distance,airbags,rollover_rating,apps,cargo_space,traction_control,rear_legroom,crash_ratings,total_score_from_contributions
0,lexus_rx_350,Lexus RX 350 AWD,7.619048,19.047619,7.142857,14.285714,8.571429,0.0,6.476190,9.52381,72.666667
1,lexus_rx_350_2009,Lexus RX 350 AWD (2009),7.619048,19.047619,7.142857,14.285714,8.571429,0.0,6.476190,9.52381,72.666667
2,honda_crv,Honda CR-V EX-L 4WD,11.809524,15.238095,7.142857,14.285714,6.114286,0.0,8.095238,9.52381,72.209524
3,mitsubishi_outlander,Mitsubishi Outlander,3.428571,19.047619,7.142857,14.285714,6.400000,0.0,9.142857,9.52381,68.971429
4,volvo_xc60,Volvo XC-60,10.285714,15.238095,7.142857,14.285714,3.314286,0.0,6.095238,9.52381,65.885714
5,acura_rdx,Acura RDX SH 4WD,9.523810,15.238095,7.142857,14.285714,1.714286,0.0,7.238095,9.52381,64.666667
6,honda_crv_2009,Honda CR-V EX-L 4WD (2009),11.809524,15.238095,7.142857,4.285714,6.114286,0.0,8.095238,9.52381,62.209524
7,toyota_rav4,Toyota RAV4,7.619048,15.238095,7.142857,4.285714,6.514286,0.0,7.904762,9.52381,58.228571
8,cadillac_srx,Cadillac SRX Luxury 4WD,11.809524,15.238095,7.142857,4.285714,2.742857,0.0,5.714286,9.52381,56.457143
9,honda_odyssey,Honda Odyssey EX-L,1.142857,19.047619,7.142857,0.000000,7.657143,0.0,9.523810,9.52381,54.038095


## 11) Stacked contribution visualization

If plotting support is available, this renders a stacked bar chart. Otherwise, use the contribution table above.


In [14]:
if HAS_DECISION_PLOTS:
    figure = DecisionAnalysisStackedBarPlotter().build_figure(decision_result)
    try:
        figure.show()
    except ValueError as error:
        if "nbformat" in str(error) or "ipython" in str(error):
            print("Plot rendering is unavailable in this notebook environment (missing nbformat/ipython).")
            print("Install with: pip install 'nbformat>=4.2.0' ipython")
            print("Falling back to contributions_df table output.")
        else:
            raise
else:
    print("decision_plots is not available in this environment. Use contributions_df as a fallback.")

## 12) Adapt this notebook to your own decision problem

Only replace domain data blocks. Keep the engine pipeline unchanged.

### Replacement checklist
1. **Criteria (`criteria_df`)**
   - Update each row's `id`, `name`, `kind`, and bounds/categories.
   - Use valid `kind` values: `continuous`, `integer`, `boolean`, `ordinal`, `categorical`.
2. **Alternatives (`raw_df`)**
   - Keep `id` and `name` columns.
   - Add one measurement column per criterion `id`.
3. **Value functions (`value_function_specs`)**
   - Provide one spec per criterion id.
   - Choose `linear_increasing`, `linear_decreasing`, `piecewise_linear`, or `mapping`.
4. **Weights (`weights_df`)**
   - Provide one row per criterion id.
   - Keep each `weight` in `[0, 100]` and ensure at least one positive weight.
5. **Feasibility rules (`feasibility_df`)**
   - Provide one row per feasibility condition your alternatives must satisfy.
   - Link each rule to a valid `criterion_id` and use a supported `predicate_kind` (`requires_true`, `not_equal`, `gte`).
   - Set `predicate_param` to `None` for `requires_true`, a comparison value for `not_equal`, and a numeric threshold for `gte`.
6. Rerun from the validation checkpoint onward.


## 13) Mini-example outline (domain-agnostic placeholder)

Use this as a quick planning frame before editing the real input cells, including feasibility constraints.


In [15]:
example_outline = {
    "problem_name": "Your decision topic",
    "criteria_examples": [
        {"id": "cost", "kind": "continuous", "interpretation": "lower is better"},
        {"id": "quality", "kind": "ordinal", "interpretation": "higher is better"},
        {"id": "has_feature_x", "kind": "boolean", "interpretation": "True preferred"},
    ],
    "alternatives_examples": [
        "Option A",
        "Option B",
        "Option C",
    ],
    "feasibility_examples": [
        {"id": "must_have_feature_x", "criterion_id": "has_feature_x", "predicate_kind": "requires_true", "predicate_param": None},
        {"id": "max_cost_gate", "criterion_id": "cost", "predicate_kind": "lte (custom)", "predicate_param": "Define custom predicate or map to built-in rules"},
    ],
    "pipeline": [
        "edit criteria_df",
        "edit raw_df",
        "edit value_function_specs",
        "edit weights_df",
        "edit feasibility_df",
        "run validation checkpoint",
        "run scoring/results/plot cells",
    ],
}

example_outline


{'problem_name': 'Your decision topic',
 'criteria_examples': [{'id': 'cost',
   'kind': 'continuous',
   'interpretation': 'lower is better'},
  {'id': 'quality', 'kind': 'ordinal', 'interpretation': 'higher is better'},
  {'id': 'has_feature_x',
   'kind': 'boolean',
   'interpretation': 'True preferred'}],
 'alternatives_examples': ['Option A', 'Option B', 'Option C'],
 'feasibility_examples': [{'id': 'must_have_feature_x',
   'criterion_id': 'has_feature_x',
   'predicate_kind': 'requires_true',
   'predicate_param': None},
  {'id': 'max_cost_gate',
   'criterion_id': 'cost',
   'predicate_kind': 'lte (custom)',
   'predicate_param': 'Define custom predicate or map to built-in rules'}],
 'pipeline': ['edit criteria_df',
  'edit raw_df',
  'edit value_function_specs',
  'edit weights_df',
  'edit feasibility_df',
  'run validation checkpoint',
  'run scoring/results/plot cells']}

## 14) Troubleshooting map (common mistakes -> where to look)

- **Missing required columns** -> Check the input-shape checkpoint error table.
- **Missing criterion columns in raw data** -> Check `raw` stage `criteria columns` message.
- **Unknown columns / ids in raw, weights, or value-function specs** -> Check table-level alignment messages.
- **Unknown feasibility `criterion_id`** -> Check `feasibility` stage `criterion_id` messages in the input-shape checkpoint table.
- **Unsupported feasibility `predicate_kind`** -> Check `feasibility` stage `predicate_kind`; use built-ins (`requires_true`, `not_equal`, `gte`) or register a custom predicate.
- **Invalid feasibility `predicate_param` (wrong type/value)** -> Check `feasibility` stage `predicate_param` and verify the rule contract (`requires_true` => `None`, `not_equal` => comparison value, `gte` => numeric).
- **Feasibility rule seems ignored in results** -> Confirm `feasibility_rules=feasibility_rules` is passed into `TotalValueModel(...)` and rerun scoring cells.
- **Kind/measurement/type incompatibility, bad bounds, bad mappings, non-numeric score outputs** -> Core construction/scoring errors now provide the source-of-truth message.
- **All zero weights or invalid weight values** -> Core `TotalValueModel` construction error will identify the issue.

Tip: Fix the first error shown, rerun the checkpoint, then continue iteratively.
